####Binance trades

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

raw_root = "s3://l1l2bucket/data/raw/binance"

catalog = "crypto_lakehouse"
bronze_schema = "bronze"

spark.sql(f"create schema if not exists {catalog}.{bronze_schema}")

print(f"Raw data: {raw_root}")
print(f"Bronze schema: {catalog}.{bronze_schema}")

display(dbutils.fs.ls(raw_root))

Raw data: s3://l1l2bucket/data/raw/binance
Bronze schema: crypto_lakehouse.bronze


path,name,size,modificationTime
s3://l1l2bucket/data/raw/binance/account/,account/,0,1790950872401
s3://l1l2bucket/data/raw/binance/convert_trades/,convert_trades/,0,1790950872401
s3://l1l2bucket/data/raw/binance/deposits/,deposits/,0,1790950872401
s3://l1l2bucket/data/raw/binance/funding_account/,funding_account/,0,1790950872401
s3://l1l2bucket/data/raw/binance/ingestion_manifest/,ingestion_manifest/,0,1790950872401
s3://l1l2bucket/data/raw/binance/p2p_buy/,p2p_buy/,0,1790950872401
s3://l1l2bucket/data/raw/binance/p2p_sell/,p2p_sell/,0,1790950872401
s3://l1l2bucket/data/raw/binance/price_history/,price_history/,0,1790950872401
s3://l1l2bucket/data/raw/binance/trades/,trades/,0,1790950872401
s3://l1l2bucket/data/raw/binance/transfers/,transfers/,0,1790950872401


In [0]:
spot_trade_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("id", LongType(), True),
    StructField("orderId", LongType(), True),
    StructField("orderListId", LongType(), True),
    StructField("price", StringType(), True),
    StructField("qty", StringType(), True),
    StructField("quoteQty", StringType(), True),
    StructField("commission", StringType(), True),
    StructField("commissionAsset", StringType(), True),
    StructField("time", LongType(), True),
    StructField("isBuyer", BooleanType(), True),
    StructField("isMaker", BooleanType(), True),
    StructField("isBestMatch", BooleanType(), True)
])

print(spot_trade_schema.simpleString())

struct<symbol:string,id:bigint,orderId:bigint,orderListId:bigint,price:string,qty:string,quoteQty:string,commission:string,commissionAsset:string,time:bigint,isBuyer:boolean,isMaker:boolean,isBestMatch:boolean>


In [0]:
spot_trades_raw_df = (
    spark.read
    .schema(spot_trade_schema)
    .option("multiline",True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/trades/")
)

spot_trades_df = (
    spot_trades_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("exchange",lit("Binance"))
    .withColumn("source_type",lit("Spot_trade"))
    .withColumn("ingested_at", current_timestamp())
)

print("Spot trade rows:", spot_trades_df.count())

display(
    spot_trades_df.orderBy(
        col("time").desc()
    )
)

Spot trade rows: 26


symbol,id,orderId,orderListId,price,qty,quoteQty,commission,commissionAsset,time,isBuyer,isMaker,isBestMatch,source_file_name,_source_file,exchange,source_type,ingested_at
BTCUSDT,6636792208,66035545193,-1,79121.50000000,0.00004000,3.16486000,0.00000004,BTC,1788114758003,true,false,true,trades_20261002T141741_197265Z_a7e61f88.json,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20261002T141741_197265Z_a7e61f88.json,Binance,Spot_trade,2026-10-02T14:21:29.303Z
BTCUSDT,6636792207,66035545193,-1,79121.50000000,0.00007000,5.53850500,0.00000007,BTC,1788114758003,true,false,true,trades_20261002T141741_197265Z_a7e61f88.json,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20261002T141741_197265Z_a7e61f88.json,Binance,Spot_trade,2026-10-02T14:21:29.303Z
BTCUSDT,6636792207,66035545193,-1,79121.50000000,0.00007000,5.53850500,0.00000007,BTC,1788114758003,true,false,true,trades_20260925T182105_429433Z_77f23558.json,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,Binance,Spot_trade,2026-10-02T14:21:29.303Z
BTCUSDT,6636792208,66035545193,-1,79121.50000000,0.00004000,3.16486000,0.00000004,BTC,1788114758003,true,false,true,trades_20260925T182105_429433Z_77f23558.json,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,Binance,Spot_trade,2026-10-02T14:21:29.303Z
ETHUSDT,4255932876,48872688120,-1,1878.41000000,0.00400000,7.51364000,0.00000400,ETH,1785943608425,true,false,true,trades_20260925T182105_922892Z_d577bdb1.json,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20260925T182105_922892Z_d577bdb1.json,Binance,Spot_trade,2026-10-02T14:21:29.303Z
ETHUSDT,4255932876,48872688120,-1,1878.41000000,0.00400000,7.51364000,0.00000400,ETH,1785943608425,true,false,true,trades_20261002T141741_657984Z_4d57cfa6.json,s3://l1l2bucket/data/raw/binance/trades/ETHUSDT/trades_20261002T141741_657984Z_4d57cfa6.json,Binance,Spot_trade,2026-10-02T14:21:29.303Z
BTCUSDT,6497931425,64430221057,-1,64266.00000000,0.00010000,6.42660000,0.00000010,BTC,1783798810981,true,false,true,trades_20261002T141741_197265Z_a7e61f88.json,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20261002T141741_197265Z_a7e61f88.json,Binance,Spot_trade,2026-10-02T14:21:29.303Z
BTCUSDT,6497931425,64430221057,-1,64266.00000000,0.00010000,6.42660000,0.00000010,BTC,1783798810981,true,false,true,trades_20260925T182105_429433Z_77f23558.json,s3://l1l2bucket/data/raw/binance/trades/BTCUSDT/trades_20260925T182105_429433Z_77f23558.json,Binance,Spot_trade,2026-10-02T14:21:29.303Z
ONDOUSDT,60681896,994044647,-1,0.33710000,18.30000000,6.16893000,0.01830000,ONDO,1783203858443,true,false,true,trades_20261002T141743_500749Z_dfb0af68.json,s3://l1l2bucket/data/raw/binance/trades/ONDOUSDT/trades_20261002T141743_500749Z_dfb0af68.json,Binance,Spot_trade,2026-10-02T14:21:29.303Z
ONDOUSDT,60681896,994044647,-1,0.33710000,18.30000000,6.16893000,0.01830000,ONDO,1783203858443,true,false,true,trades_20260925T182107_841454Z_e13b5606.json,s3://l1l2bucket/data/raw/binance/trades/ONDOUSDT/trades_20260925T182107_841454Z_e13b5606.json,Binance,Spot_trade,2026-10-02T14:21:29.303Z


In [0]:
spot_rows_before = spot_trades_df.count()


spot_trades_bronze_df = (
    spot_trades_df
    .dropDuplicates([
        "symbol",
        "id"
    ])
)


spot_rows_after = spot_trades_bronze_df.count()
spot_duplicates_removed = spot_rows_before - spot_rows_after


print("Rows before deduplication:", spot_rows_before)
print("Rows after deduplication:", spot_rows_after)
print("Duplicate rows removed:", spot_duplicates_removed)


display(
    spot_trades_bronze_df
    .groupBy("symbol")
    .count()
    .orderBy("symbol")
)


Rows before deduplication: 26
Rows after deduplication: 13
Duplicate rows removed: 13


symbol,count
BTCUSDT,4
ETHUSDT,2
LINKUSDT,2
ONDOUSDT,1
SOLUSDT,3
SUIUSDT,1


In [0]:
(
    spot_trades_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema",True)
    .saveAsTable(f"{catalog}.{bronze_schema}.binance_spot_trades")
)

####Binance Convert trade

In [0]:
convert_trade_schema = StructType([
    StructField("quoteId", StringType(), True),
    StructField("orderId", LongType(), True),
    StructField("orderStatus", StringType(), True),
    StructField("fromAsset", StringType(), True),
    StructField("fromAmount", StringType(), True),
    StructField("toAsset", StringType(), True),
    StructField("toAmount", StringType(), True),
    StructField("ratio", StringType(), True),
    StructField("inverseRatio", StringType(), True),
    StructField("createTime", LongType(), True),
    StructField("orderType", StringType(), True),
    StructField("side", StringType(), True),
    StructField("walletType", StringType(), True)
])

print(convert_trade_schema.simpleString())

struct<quoteId:string,orderId:bigint,orderStatus:string,fromAsset:string,fromAmount:string,toAsset:string,toAmount:string,ratio:string,inverseRatio:string,createTime:bigint,orderType:string,side:string,walletType:string>


In [0]:
convert_trades_raw_df = (
    spark.read
    .schema(convert_trade_schema)
    .option("multiline",True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/convert_trades/")
)

convert_trades_df = (
    convert_trades_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("exchange",lit("Binance"))
    .withColumn("source_type",lit("Convert_trade"))
    .withColumn("ingested_at",current_timestamp())
)

print("convert trade rows:", convert_trades_df.count())

display(
    convert_trades_df.orderBy(
        col("createTime").desc()
    )
)



convert trade rows: 6


quoteId,orderId,orderStatus,fromAsset,fromAmount,toAsset,toAmount,ratio,inverseRatio,createTime,orderType,side,walletType,source_file_name,_source_file,exchange,source_type,ingested_at
T69B3306C_B0000000000038786,2221766670752487619,SUCCESS,USDT,3.54,SOL,0.03993758,0.0112818,88.6383,1773383682256,MARKET,BUY,FUNDING,convert_trades_20261002T141749_796495Z_29cde572.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20261002T141749_796495Z_29cde572.json,Binance,Convert_trade,2026-10-02T14:21:42.156Z
T69B3306C_B0000000000038786,2221766670752487619,SUCCESS,USDT,3.54,SOL,0.03993758,0.0112818,88.6383,1773383682256,MARKET,BUY,FUNDING,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:42.156Z
T69B3306C_B0000000000036979,2221756019233545619,SUCCESS,USDT,4.04000058,SOL,0.04526871,0.0112051,89.2449,1773382442380,MARKET,BUY,SPOT,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:42.156Z
T69B3306C_B0000000000036979,2221756019233545619,SUCCESS,USDT,4.04000058,SOL,0.04526871,0.0112051,89.2449,1773382442380,MARKET,BUY,SPOT,convert_trades_20261002T141749_796495Z_29cde572.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20261002T141749_796495Z_29cde572.json,Binance,Convert_trade,2026-10-02T14:21:42.156Z
T69B1DEEB_B000000000006A953,2221355118396379831,SUCCESS,USDT,1.95,SOL,0.02244853,0.0115121,86.8654,1773335771268,MARKET,BUY,FUNDING,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:42.156Z
T69B1DEEB_B000000000006A953,2221355118396379831,SUCCESS,USDT,1.95,SOL,0.02244853,0.0115121,86.8654,1773335771268,MARKET,BUY,FUNDING,convert_trades_20261002T141749_796495Z_29cde572.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20261002T141749_796495Z_29cde572.json,Binance,Convert_trade,2026-10-02T14:21:42.156Z


In [0]:
convert_rows_before = convert_trades_df.count()

convert_trades_bronze_df = (
    convert_trades_df
    .dropDuplicates([
        "orderId"
    ])
)

convert_rows_after = convert_trades_bronze_df.count()
duplicates_removed = convert_rows_before - convert_rows_after

print("Rows before deduplication:", convert_rows_before)
print("Rows after deduplication:", convert_rows_after)
print("Duplicate rows removed:", duplicates_removed)

display(
    convert_trades_bronze_df)

Rows before deduplication: 6
Rows after deduplication: 3
Duplicate rows removed: 3


quoteId,orderId,orderStatus,fromAsset,fromAmount,toAsset,toAmount,ratio,inverseRatio,createTime,orderType,side,walletType,source_file_name,_source_file,exchange,source_type,ingested_at
T69B1DEEB_B000000000006A953,2221355118396379831,SUCCESS,USDT,1.95,SOL,0.02244853,0.0115121,86.8654,1773335771268,MARKET,BUY,FUNDING,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:43.846Z
T69B3306C_B0000000000038786,2221766670752487619,SUCCESS,USDT,3.54,SOL,0.03993758,0.0112818,88.6383,1773383682256,MARKET,BUY,FUNDING,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:43.846Z
T69B3306C_B0000000000036979,2221756019233545619,SUCCESS,USDT,4.04000058,SOL,0.04526871,0.0112051,89.2449,1773382442380,MARKET,BUY,SPOT,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:43.846Z


In [0]:
(
    convert_trades_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_convert_trades"
    )
)

In [0]:
print(
    "Saved rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_convert_trades"
    ).count()
)
display(
    spark.table(
        f"{catalog}.{bronze_schema}.binance_convert_trades"
    )
)

Saved rows: 3


quoteId,orderId,orderStatus,fromAsset,fromAmount,toAsset,toAmount,ratio,inverseRatio,createTime,orderType,side,walletType,source_file_name,_source_file,exchange,source_type,ingested_at
T69B1DEEB_B000000000006A953,2221355118396379831,SUCCESS,USDT,1.95,SOL,0.02244853,0.0115121,86.8654,1773335771268,MARKET,BUY,FUNDING,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:46.415Z
T69B3306C_B0000000000038786,2221766670752487619,SUCCESS,USDT,3.54,SOL,0.03993758,0.0112818,88.6383,1773383682256,MARKET,BUY,FUNDING,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:46.415Z
T69B3306C_B0000000000036979,2221756019233545619,SUCCESS,USDT,4.04000058,SOL,0.04526871,0.0112051,89.2449,1773382442380,MARKET,BUY,SPOT,convert_trades_20260925T182111_033474Z_98533535.json,s3://l1l2bucket/data/raw/binance/convert_trades/convert_trades_20260925T182111_033474Z_98533535.json,Binance,Convert_trade,2026-10-02T14:21:46.415Z


####Binance p2p

In [0]:
p2p_schema = StructType([
    StructField("orderNumber", StringType(), True),
    StructField("advNo", StringType(), True),
    StructField("tradeType", StringType(), True),
    StructField("asset", StringType(), True),
    StructField("fiat", StringType(), True),
    StructField("fiatSymbol", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("totalPrice", StringType(), True),
    StructField("unitPrice", StringType(), True),
    StructField("orderStatus", StringType(), True),
    StructField("createTime", LongType(), True),
    StructField("takerCommissionRate", StringType(), True),
    StructField("takerCommission", StringType(), True),
    StructField("takerAmount", StringType(), True),
    StructField("counterPartNickName", StringType(), True),
    StructField("payMethodName", StringType(), True),
    StructField("additionalKycVerify", LongType(), True)
])

print(p2p_schema.simpleString())

struct<orderNumber:string,advNo:string,tradeType:string,asset:string,fiat:string,fiatSymbol:string,amount:string,totalPrice:string,unitPrice:string,orderStatus:string,createTime:bigint,takerCommissionRate:string,takerCommission:string,takerAmount:string,counterPartNickName:string,payMethodName:string,additionalKycVerify:bigint>


In [0]:
p2p_buy_df = (
    spark.read
    .schema(p2p_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup",True)
    .json(f"{raw_root}/p2p_buy/")
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file"),
    )
)

p2p_sell_df = (
    spark.read
    .schema(p2p_schema)
    .option("multiLine",True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/p2p_sell/")
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file"),
    )
)

p2p_trades_df = (
    p2p_buy_df
    .unionByName(p2p_sell_df)
    .withColumn("exchange", lit("Binance"))
    .withColumn("source_type", lit("P2P_trade"))
    .withColumn("ingested_at", current_timestamp())
)

print("P2P BUY rows:", p2p_buy_df.count())
print("P2P SELL rows:", p2p_sell_df.count())
print("Total P2P rows:", p2p_trades_df.count())

display(
    p2p_trades_df.orderBy(
        col("createTime").desc()
    )
)

P2P BUY rows: 30
P2P SELL rows: 0
Total P2P rows: 30


orderNumber,advNo,tradeType,asset,fiat,fiatSymbol,amount,totalPrice,unitPrice,orderStatus,createTime,takerCommissionRate,takerCommission,takerAmount,counterPartNickName,payMethodName,additionalKycVerify,source_file_name,_source_file,exchange,source_type,ingested_at
22927112581471313920,12927111932035825664,BUY,USDT,INR,₹,5.33000000,550.00000000,103,COMPLETED,1788034868551,0,0.06,5.27,Dim***,UPI,0,p2p_buy_20261002T141753_069106Z_62c0c62d.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20261002T141753_069106Z_62c0c62d.json,Binance,P2P_trade,2026-10-02T14:22:02.649Z
22927112581471313920,12927111932035825664,BUY,USDT,INR,₹,5.33000000,550.00000000,103,COMPLETED,1788034868551,0,0.06,5.27,Dim***,UPI,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:02.649Z
22924149347226902528,12908479299326263296,BUY,USDT,INR,₹,3.86000000,400.00000000,103.5,COMPLETED,1787328378475,0,0.06,3.8,P2P***,UPI,0,p2p_buy_20261002T141753_069106Z_62c0c62d.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20261002T141753_069106Z_62c0c62d.json,Binance,P2P_trade,2026-10-02T14:22:02.649Z
22924149347226902528,12908479299326263296,BUY,USDT,INR,₹,3.86000000,400.00000000,103.5,COMPLETED,1787328378475,0,0.06,3.8,P2P***,UPI,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:02.649Z
22918338709083369472,12828747426765901824,BUY,USDT,INR,₹,3.88000000,400.00000000,102.94,COMPLETED,1785943014361,0,0.06,3.82,DL8***,UPI,0,p2p_buy_20261002T141753_069106Z_62c0c62d.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20261002T141753_069106Z_62c0c62d.json,Binance,P2P_trade,2026-10-02T14:22:02.649Z
22918338709083369472,12828747426765901824,BUY,USDT,INR,₹,3.88000000,400.00000000,102.94,COMPLETED,1785943014361,0,0.06,3.82,DL8***,UPI,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:02.649Z
22910348341135761408,12905233475576479744,BUY,USDT,INR,₹,3.42000000,350.00000000,102.3,COMPLETED,1784037962235,0,0.06,3.36,THA***,UPIQRCode,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:02.649Z
22910348341135761408,12905233475576479744,BUY,USDT,INR,₹,3.42000000,350.00000000,102.3,COMPLETED,1784037962235,0,0.06,3.36,THA***,UPIQRCode,0,p2p_buy_20261002T141753_069106Z_62c0c62d.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20261002T141753_069106Z_62c0c62d.json,Binance,P2P_trade,2026-10-02T14:22:02.649Z
22909209175754317824,12908121022744678400,BUY,USDT,INR,₹,3.42000000,350.00000000,102.2,COMPLETED,1783766364047,0,0.06,3.36,No-***,UPIQRCode,0,p2p_buy_20260925T182113_602514Z_e1426d81.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20260925T182113_602514Z_e1426d81.json,Binance,P2P_trade,2026-10-02T14:22:02.649Z
22909209175754317824,12908121022744678400,BUY,USDT,INR,₹,3.42000000,350.00000000,102.2,COMPLETED,1783766364047,0,0.06,3.36,No-***,UPIQRCode,0,p2p_buy_20261002T141753_069106Z_62c0c62d.json,s3://l1l2bucket/data/raw/binance/p2p_buy/p2p_buy_20261002T141753_069106Z_62c0c62d.json,Binance,P2P_trade,2026-10-02T14:22:02.649Z


In [0]:
p2p_rows_before = p2p_trades_df.count()

p2p_trades_bronze_df = (
    p2p_trades_df
    .dropDuplicates([
        "orderNumber"
    ])
)

p2p_rows_after = p2p_trades_bronze_df.count()

total_duplicates_removed = p2p_rows_before - p2p_rows_after

print("P2P rows before deduplication:", p2p_rows_before)
print("P2P rows after deduplication:", p2p_rows_after)
print("Total duplicate P2P rows removed:", total_duplicates_removed)

display(
    p2p_trades_bronze_df
    .groupBy("tradeType")
    .count()
)

P2P rows before deduplication: 30
P2P rows after deduplication: 15
Total duplicate P2P rows removed: 15


tradeType,count
BUY,15


In [0]:
(
    p2p_trades_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_p2p_trades")
)

print(
    "Saved P2P rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_p2p_trades"
    ).count()
)

Saved P2P rows: 15


####Binance account

In [0]:
commission_rates_schema = StructType([
    StructField("maker", StringType(), True),
    StructField("taker", StringType(), True),
    StructField("buyer", StringType(), True),
    StructField("seller", StringType(), True)
])


balance_schema = StructType([
    StructField("asset", StringType(), True),
    StructField("free", StringType(), True),
    StructField("locked", StringType(), True)
])


account_schema = StructType([
    StructField("makerCommission", LongType(), True),
    StructField("takerCommission", LongType(), True),
    StructField("buyerCommission", LongType(), True),
    StructField("sellerCommission", LongType(), True),
    StructField("commissionRates", commission_rates_schema, True),
    StructField("canTrade", BooleanType(), True),
    StructField("canWithdraw", BooleanType(), True),
    StructField("canDeposit", BooleanType(), True),
    StructField("brokered", BooleanType(), True),
    StructField("requireSelfTradePrevention", BooleanType(), True),
    StructField("preventSor", BooleanType(), True),
    StructField("updateTime", LongType(), True),
    StructField("accountType", StringType(), True),
    StructField(
        "balances",
        ArrayType(balance_schema),
        True
    ),
    StructField(
        "permissions",
        ArrayType(StringType()),
        True
    ),
    StructField("uid", LongType(), True)
])


print(account_schema.simpleString())

struct<makerCommission:bigint,takerCommission:bigint,buyerCommission:bigint,sellerCommission:bigint,commissionRates:struct<maker:string,taker:string,buyer:string,seller:string>,canTrade:boolean,canWithdraw:boolean,canDeposit:boolean,brokered:boolean,requireSelfTradePrevention:boolean,preventSor:boolean,updateTime:bigint,accountType:string,balances:array<struct<asset:string,free:string,locked:string>>,permissions:array<string>,uid:bigint>


In [0]:
account_raw_df = (
    spark.read
    .schema(account_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/account/")
)


account_df = (
    account_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Binance"))
    .withColumn("source_type", lit("Account_snapshot"))
    .withColumn("ingested_at", current_timestamp())
)


print(
    "Account snapshot rows:",
    account_df.count()
)


display(
    account_df.select(
        "uid",
        "accountType",
        "canTrade",
        "canDeposit",
        "canWithdraw",
        "updateTime",
        size("balances").alias("number_of_assets"),
        "source_file_name",
        "ingested_at"
    )
)

Account snapshot rows: 2


uid,accountType,canTrade,canDeposit,canWithdraw,updateTime,number_of_assets,source_file_name,ingested_at
1225138972,SPOT,true,true,true,1788114758003,852,account_20261002T141739_738195Z_b4941cd6.json,2026-10-02T14:22:16.848Z
1225138972,SPOT,true,true,true,1788114758003,845,account_20260925T182104_128404Z_4e8bc291.json,2026-10-02T14:22:16.848Z


In [0]:
account_rows_before = account_df.count()


account_bronze_df = (
    account_df
    .dropDuplicates([
        "_source_file"
    ])
)

account_rows_after = account_bronze_df.count()


print(
    "Account rows before deduplication:",
    account_rows_before
)

print(
    "Account rows after deduplication:",
    account_rows_after
)

print(
    "Duplicate account rows removed:",
    account_rows_before - account_rows_after
)

Account rows before deduplication: 2
Account rows after deduplication: 2
Duplicate account rows removed: 0


In [0]:
(
    account_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_account_snapshots"
    )
)

print(
    "Saved account snapshots:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_account_snapshots"
    ).count()
)

Saved account snapshots: 2


####Binance Deposits

In [0]:
deposit_schema = StructType([
    StructField("id", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("coin", StringType(), True),
    StructField("network", StringType(), True),
    StructField("status", LongType(), True),
    StructField("address", StringType(), True),
    StructField("addressTag", StringType(), True),
    StructField("txId", StringType(), True),
    StructField("sourceAddress", StringType(), True),
    StructField("insertTime", LongType(), True),
    StructField("completeTime", LongType(), True),
    StructField("transferType", LongType(), True),
    StructField("confirmTimes", StringType(), True),
    StructField("unlockConfirm", LongType(), True),
    StructField("walletType", LongType(), True),
    StructField("travelRuleStatus", LongType(), True)
])


print(deposit_schema.simpleString())

struct<id:string,amount:string,coin:string,network:string,status:bigint,address:string,addressTag:string,txId:string,sourceAddress:string,insertTime:bigint,completeTime:bigint,transferType:bigint,confirmTimes:string,unlockConfirm:bigint,walletType:bigint,travelRuleStatus:bigint>


In [0]:
deposits_raw_df = (
    spark.read
    .schema(deposit_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/deposits/")
)


deposits_df = (
    deposits_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Binance"))
    .withColumn("source_type", lit("Deposit"))
    .withColumn("ingested_at", current_timestamp())
)


print(
    "Deposit rows:",
    deposits_df.count()
)


display(
    deposits_df.orderBy(
        col("insertTime").desc()
    )
)

Deposit rows: 0


id,amount,coin,network,status,address,addressTag,txId,sourceAddress,insertTime,completeTime,transferType,confirmTimes,unlockConfirm,walletType,travelRuleStatus,source_file_name,_source_file,exchange,source_type,ingested_at


In [0]:
deposits_bronze_df = (
    deposits_df
    .dropDuplicates([
        "id"
    ])
)


(
    deposits_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_deposits"
    )
)


print(
    "Saved deposit rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_deposits"
    ).count()
)

Saved deposit rows: 0


####Binance withdrawals

In [0]:
withdrawal_schema = StructType([
    StructField("id", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("transactionFee", StringType(), True),
    StructField("coin", StringType(), True),
    StructField("status", LongType(), True),
    StructField("address", StringType(), True),
    StructField("txId", StringType(), True),
    StructField("applyTime", StringType(), True),
    StructField("network", StringType(), True),
    StructField("transferType", LongType(), True),
    StructField("withdrawOrderId", StringType(), True),
    StructField("info", StringType(), True),
    StructField("confirmNo", LongType(), True),
    StructField("walletType", LongType(), True),
    StructField("txKey", StringType(), True),
    StructField("completeTime", StringType(), True)
])


print(withdrawal_schema.simpleString())

struct<id:string,amount:string,transactionFee:string,coin:string,status:bigint,address:string,txId:string,applyTime:string,network:string,transferType:bigint,withdrawOrderId:string,info:string,confirmNo:bigint,walletType:bigint,txKey:string,completeTime:string>


In [0]:
withdrawals_raw_df = (
    spark.read
    .schema(withdrawal_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/withdrawals/")
)


withdrawals_df = (
    withdrawals_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Binance"))
    .withColumn("source_type", lit("Withdrawal"))
    .withColumn("ingested_at", current_timestamp())
)


print(
    "Withdrawal rows:",
    withdrawals_df.count()
)


display(
    withdrawals_df.orderBy(
        col("applyTime").desc()
    )
)

Withdrawal rows: 2


id,amount,transactionFee,coin,status,address,txId,applyTime,network,transferType,withdrawOrderId,info,confirmNo,walletType,txKey,completeTime,source_file_name,_source_file,exchange,source_type,ingested_at
3f2234d4647746d4ac478d1e8d792855,0.10665482,0.001,SOL,6,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,kGWB1m8K4ZvZYRGncfZMv3A1okHhKMrwacGfUn9FuEnLButrU92rZFj2xqwV48ftYwJQV5qAYwUZrCeecaaSvfU,2026-03-13 06:39:17,SOL,0,null,"blockhash:FsvRc4r4qqAZApYTGEKmy8QhVGH7wt2fsXUuyx3cbgBa,nonceAccount:3kQUuRN7wu6fhfpxpKjszWKXpeE1kav58aoMsXZ9sciL",2,0,,2026-03-13 06:41:06,withdrawals_20260925T182120_839679Z_eb18a75d.json,s3://l1l2bucket/data/raw/binance/withdrawals/withdrawals_20260925T182120_839679Z_eb18a75d.json,Binance,Withdrawal,2026-10-02T14:22:39.947Z
3f2234d4647746d4ac478d1e8d792855,0.10665482,0.001,SOL,6,FkX7NDEzwYNT7rjuEQaeL3MAP2mYGPLZCGuR7hKBXMK7,kGWB1m8K4ZvZYRGncfZMv3A1okHhKMrwacGfUn9FuEnLButrU92rZFj2xqwV48ftYwJQV5qAYwUZrCeecaaSvfU,2026-03-13 06:39:17,SOL,0,null,"blockhash:FsvRc4r4qqAZApYTGEKmy8QhVGH7wt2fsXUuyx3cbgBa,nonceAccount:3kQUuRN7wu6fhfpxpKjszWKXpeE1kav58aoMsXZ9sciL",2,0,,2026-03-13 06:41:06,withdrawals_20261002T141801_944582Z_a8f522b8.json,s3://l1l2bucket/data/raw/binance/withdrawals/withdrawals_20261002T141801_944582Z_a8f522b8.json,Binance,Withdrawal,2026-10-02T14:22:39.947Z


In [0]:
withdrawals_bronze_df = (
    withdrawals_df
    .dropDuplicates([
        "id"
    ])
)


print(
    "Duplicate withdrawal rows removed:",
    withdrawals_df.count() - withdrawals_bronze_df.count()
)


(
    withdrawals_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_withdrawals"
    )
)


print(
    "Saved withdrawal rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_withdrawals"
    ).count()
)

Duplicate withdrawal rows removed: 1
Saved withdrawal rows: 1


####Binance transfers

In [0]:
transfer_schema = StructType([
    StructField("tranId", LongType(), True),
    StructField("asset", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("type", StringType(), True),
    StructField("status", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("transferType", StringType(), True)
])


print(transfer_schema.simpleString())

struct<tranId:bigint,asset:string,amount:string,type:string,status:string,timestamp:bigint,transferType:string>


In [0]:
transfers_raw_df = (
    spark.read
    .schema(transfer_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/transfers/")
)


transfers_df = (
    transfers_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Binance"))
    .withColumn("source_type", lit("Transfer"))
    .withColumn("ingested_at", current_timestamp())
)


print(
    "Transfer rows:",
    transfers_df.count()
)


display(
    transfers_df.orderBy(
        col("timestamp").desc()
    )
)

Transfer rows: 0


tranId,asset,amount,type,status,timestamp,transferType,source_file_name,_source_file,exchange,source_type,ingested_at


In [0]:
transfers_bronze_df = (
    transfers_df
    .dropDuplicates([
        "tranId"
    ])
)


print(
    "Duplicate transfer rows removed:",
    transfers_df.count() - transfers_bronze_df.count()
)


(
    transfers_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_transfers"
    )
)


print(
    "Saved transfer rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_transfers"
    ).count()
)

Duplicate transfer rows removed: 0
Saved transfer rows: 0


####Binance price history

In [0]:
kline_file_schema = ArrayType(
    ArrayType(
        StringType()
    )
)


kline_field_names = [
    "open_time",
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "unused_field"
]


print(kline_file_schema.simpleString())
print(kline_field_names)

array<array<string>>
['open_time', 'open_price', 'high_price', 'low_price', 'close_price', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_volume', 'taker_buy_quote_volume', 'unused_field']


In [0]:
price_files_df = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", True)
    .load(f"{raw_root}/price_history/")
)


price_klines_raw_df = (
    price_files_df
    .withColumn(
        "symbol",
        regexp_extract(
            col("path"),
            r"/price_history/([^/]+)/",
            1
        )
    )
    .withColumn(
        "klines",
        from_json(
            decode(col("content"), "UTF-8"),
            kline_file_schema
        )
    )
    .select(
        "symbol",

        col("path").alias("_source_file"),

        to_timestamp(
            regexp_extract(
                col("path"),
                r"price_history_(\d{8}T\d{6}_\d{6}Z)_",
                1
            ),
            "yyyyMMdd'T'HHmmss_SSSSSS'Z'"
        ).alias("source_file_saved_at"),

        col("modificationTime").alias(
            "source_file_modified_at"
        ),

        explode("klines").alias("kline")
    )
)


print(
    "Candlestick rows:",
    price_klines_raw_df.count()
)


display(
    price_klines_raw_df.select(
        "symbol",
        "_source_file",
        "source_file_saved_at",
        "source_file_modified_at"
    ).limit(10)
)

Candlestick rows: 4560


symbol,_source_file,source_file_saved_at,source_file_modified_at
BTCUSDT,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.901Z,2026-10-02T14:18:03.000Z
BTCUSDT,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.901Z,2026-10-02T14:18:03.000Z
BTCUSDT,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.901Z,2026-10-02T14:18:03.000Z
BTCUSDT,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.901Z,2026-10-02T14:18:03.000Z
BTCUSDT,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.901Z,2026-10-02T14:18:03.000Z
BTCUSDT,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.901Z,2026-10-02T14:18:03.000Z
BTCUSDT,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.901Z,2026-10-02T14:18:03.000Z
BTCUSDT,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.901Z,2026-10-02T14:18:03.000Z
BTCUSDT,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.901Z,2026-10-02T14:18:03.000Z
BTCUSDT,s3://l1l2bucket/data/raw/binance/price_history/BTCUSDT/price_history_20261002T141803_901793Z_4e396c56.json,2026-10-02T14:18:03.901Z,2026-10-02T14:18:03.000Z


In [0]:
price_files_df.printSchema()

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)



In [0]:
kline_columns = [
    col("kline")[index].alias(name)
    for index, name in enumerate(kline_field_names)
]


price_history_df = (
    price_klines_raw_df
    .select(
        "symbol",
        *kline_columns,
        "_source_file",
        "source_file_saved_at",
        "source_file_modified_at"
    )
    .withColumn(
        "exchange",
        lit("Binance")
    )
    .withColumn(
        "source_type",
        lit("Daily_price")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
)


print(
    "Price-history rows:",
    price_history_df.count()
)

print(
    "Rows missing file save time:",
    price_history_df
    .filter(col("source_file_saved_at").isNull())
    .count()
)

Price-history rows: 4560
Rows missing file save time: 0


In [0]:
from pyspark.sql.window import Window


latest_price_file_window = (
    Window
    .partitionBy(
        "symbol",
        "open_time"
    )
    .orderBy(
        col("source_file_saved_at").desc(),
        col("_source_file").desc()
    )
)


price_history_bronze_df = (
    price_history_df
    .withColumn(
        "row_number",
        row_number().over(latest_price_file_window)
    )
    .filter(col("row_number") == 1)
    .drop("row_number")
)


print(
    "Duplicate candles removed:",
    price_history_df.count()
    - price_history_bronze_df.count()
)


(
    price_history_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_price_history"
    )
)


print(
    "Saved candlestick rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_price_history"
    ).count()
)

Duplicate candles removed: 1608
Saved candlestick rows: 2952


####Binance funding wallet

In [0]:
funding_balance_schema = StructType([
    StructField("asset", StringType(), True),
    StructField("free", StringType(), True),
    StructField("locked", StringType(), True),
    StructField("freeze", StringType(), True),
    StructField("withdrawing", StringType(), True),
    StructField("btcValuation", StringType(), True)
])

funding_account_schema = StructType([
    StructField("walletType", StringType(), True),
    StructField("capturedAt", StringType(), True),

    StructField(
        "balances",
        ArrayType(
            funding_balance_schema,
            True
        ),
        True
    )
])

In [0]:
funding_account_raw_df = (
    spark.read
    .schema(funding_account_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(
        f"{raw_root}/funding_account/"
    )
)

funding_account_bronze_df = (
    funding_account_raw_df
    .select(
        "*",

        col("_metadata.file_name").alias(
            "source_file_name"
        ),

        col("_metadata.file_path").alias(
            "_source_file"
        )
    )
    .withColumn(
        "exchange",
        lit("Binance")
    )
    .withColumn(
        "source_type",
        lit("Funding_account")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
)

print(
    "Funding account snapshot rows:",
    funding_account_bronze_df.count()
)

funding_account_bronze_df.printSchema()

display(funding_account_bronze_df)

Funding account snapshot rows: 2
root
 |-- walletType: string (nullable = true)
 |-- capturedAt: string (nullable = true)
 |-- balances: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- asset: string (nullable = true)
 |    |    |-- free: string (nullable = true)
 |    |    |-- locked: string (nullable = true)
 |    |    |-- freeze: string (nullable = true)
 |    |    |-- withdrawing: string (nullable = true)
 |    |    |-- btcValuation: string (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- exchange: string (nullable = false)
 |-- source_type: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)



walletType,capturedAt,balances,source_file_name,_source_file,exchange,source_type,ingested_at
FUNDING,2026-09-25T18:21:04.576325+00:00,List(),funding_account_20260925T182104_949722Z_6b29dd5e.json,s3://l1l2bucket/data/raw/binance/funding_account/funding_account_20260925T182104_949722Z_6b29dd5e.json,Binance,Funding_account,2026-10-02T14:23:31.822Z
FUNDING,2026-10-02T14:17:40.159991+00:00,List(),funding_account_20261002T141740_508437Z_7aee3462.json,s3://l1l2bucket/data/raw/binance/funding_account/funding_account_20261002T141740_508437Z_7aee3462.json,Binance,Funding_account,2026-10-02T14:23:31.822Z


In [0]:
(
    funding_account_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_funding_account_snapshots"
    )
)

saved_funding_rows = spark.table(
    f"{catalog}.{bronze_schema}.binance_funding_account_snapshots"
).count()

print(
    "Saved Funding account snapshots:",
    saved_funding_rows
)

Saved Funding account snapshots: 2


In [0]:
bronze_tables = [
    "binance_spot_trades",
    "binance_convert_trades",
    "binance_p2p_trades",
    "binance_account_snapshots",
    "binance_deposits",
    "binance_withdrawals",
    "binance_transfers",
    "binance_price_history",
    "binance_funding_account_snapshots"
]


for table_name in bronze_tables:

    full_table_name = (
        f"{catalog}.{bronze_schema}.{table_name}"
    )

    row_count = (
        spark.table(full_table_name)
        .count()
    )

    print(
        f"{table_name}: {row_count} rows"
    )

binance_spot_trades: 13 rows
binance_convert_trades: 3 rows
binance_p2p_trades: 15 rows
binance_account_snapshots: 2 rows
binance_deposits: 0 rows
binance_withdrawals: 1 rows
binance_transfers: 0 rows
binance_price_history: 2952 rows
binance_funding_account_snapshots: 2 rows
